Working copy with cell outputs: [`compare_analysis_runs.local.ipynb`](compare_analysis_runs.local.ipynb) (gitignored). This file is a clean template; make git edits here, and do day-to-day analysis in `.local`.

# Comparing analysis runs by design success

The notebook reads artifacts from `article_results/analysis_runs/<run_id>__criteria_v3/` (if `metrics.csv` is present), otherwise `__criteria_v2/` (publication AND-success), otherwise unversioned. `build_context(..., criteria="v3"|"v2")` selects the version; campaigns are joined by id without the suffix.

- **one-shot publication (`pass_final`)**: pooled (HA < 2.0, Cα < 1.0, global HA < 2.0; ProteinZen paper §3.2.1) **AND** every motif group
- **pooled legacy (`pass_pooled_legacy`)**: diagnostic only; no Foldseek
- **PMPNN+ESMFold**: same motif cuts; global BB < 2.0; Unique is Foldseek of AND-success **design** backbones (`pmpnn_esmfold` in `unique_success_all_groups.json`)
- **unique success**: Foldseek clusters of AND-success **design** backbones (`unique_success_all_groups.json`, TM 0.5, per-task; `one_shot` and `pmpnn_esmfold`)

Column schema: [`metrics_schema.md`](../analysis_runs/metrics_schema.md).

In [ ]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

# Import the module from scripts/analysis.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "scripts").exists():
    for parent in REPO_ROOT.parents:
        if (parent / "scripts").exists():
            REPO_ROOT = parent
            break

sys.path.insert(0, str(REPO_ROOT))

from scripts.analysis.compare_analysis_runs_lib import (
    TLOCK05_SE3_FULL_SOURCE_IDS,
    build_context,
    build_multimotif_summary,
    build_task_pivot,
    best_run_per_task,
    display_columns,
    export_tables,
    plot_global_rmsd_distribution,
    plot_rmsd_distribution,
    plot_success_rates,
    plot_unique_success,
)

CRITERIA = "v3"  # "v2" pins AND-success; "v3" falls back to v2 if no metrics.csv
RUN_IDS = ["one_motif_proteinzen", *TLOCK05_SE3_FULL_SOURCE_IDS]
CTX = build_context(REPO_ROOT, run_ids=RUN_IDS, criteria=CRITERIA)
print(f"Analysis root: {CTX['analysis_root']}")
print(f"Criteria pin: {CRITERIA!r}")
for run_dir in CTX["run_dirs"]:
    print(f"  {run_dir.name}")

In [ ]:
RUN_DIRS = CTX["run_dirs"]
print(f"Found runs: {len(RUN_DIRS)}")
for run_dir in RUN_DIRS:
    print(f"  - {run_dir.name}")

In [ ]:
RUNS = CTX["runs"]
METRICS = CTX["metrics"]
TASK_SUCCESS = CTX["task_success"]
MULTIMOTIF = CTX["multimotif"]

print(f"metrics rows: {len(METRICS):,}")
print(f"task_success rows: {len(TASK_SUCCESS):,}")
print(f"multimotif rows: {len(MULTIMOTIF):,}")

## Summary by run

Aggregates over all tasks in each run: success rate, unique success, mean RMSD.

In [ ]:
RUN_SUMMARY = CTX["run_summary"]
RUN_SUMMARY[display_columns()]

## Success by task (task_success.csv)

In [ ]:
TASK_PIVOT = build_task_pivot(TASK_SUCCESS)
if TASK_PIVOT.empty:
    print("task_success.csv was not found in any run.")
else:
    TASK_PIVOT

## Visualization: success rate by run

In [ ]:
plot_success_rates(RUN_SUMMARY)

In [ ]:
plot_unique_success(RUN_SUMMARY)

## RMSD distribution (metrics.csv)

Comparison of motif HA RMSD distributions for designs that passed and designs that did not.

In [ ]:
PROTOCOL = "pmpnn_esmfold"  # or "one_shot"
plot_rmsd_distribution(METRICS, RUN_SUMMARY, protocol=PROTOCOL)
plot_global_rmsd_distribution(METRICS, RUN_SUMMARY, protocol=PROTOCOL)

## Multi-motif metrics (if present)

Fraction of motif groups recovered relative to the native PDB (group_pass_rate from the slices summary).

In [ ]:
MULTIMOTIF_SUMMARY = build_multimotif_summary(RUNS)
if MULTIMOTIF_SUMMARY.empty:
    print("multimotif_slices.summary.csv is empty or missing in every run.")
else:
    display(MULTIMOTIF_SUMMARY)

## Head-to-head: best run per task

For each task, show the run with the highest PMPNN success rate.

In [ ]:
BEST_PER_TASK = best_run_per_task(TASK_SUCCESS)
if BEST_PER_TASK.empty:
    print("No task_success data for the head-to-head comparison.")
else:
    BEST_PER_TASK

## Export the summary table

In [ ]:
saved = export_tables(CTX["repo_root"], RUN_SUMMARY, TASK_SUCCESS)
for name, path in saved.items():
    print(f"Saved {name}: {path}")